# Negative-Control Responses Merge (n=30, six methods)

Merges the negative-control responses generated by notebooks 11 (T21 RAG,
GPT-5.4, Claude, Nemotron), 13 (MedRAG) and 14 (PaperQA2) into ONE Excel
sheet: one row per question, one column per model, each cell holding that
model's full response text. The negative-control counterpart of notebook 12
(which merges the positive-control responses).

Merge only - no responses are generated here and no classification is done;
this notebook just reads the three checkpoint JSONs and writes the merged
files. Run with the dschat kernel, after notebooks 11, 13 and 14.

## Cell 1 - Configuration

In [4]:
# ---- Cell 1: configuration ----
import json, os, re, shutil
from datetime import datetime
from pathlib import Path

import pandas as pd

NB_DIR   = Path.cwd()
ANALYSIS = NB_DIR.parent if NB_DIR.name == 'nbs' else NB_DIR

EXCEL_PATH = ANALYSIS / 'input' / 'Revised_DownSyndrome_60_Questions_Final_linked.xlsx'
OUT_DIR    = ANALYSIS / 'output' / 'ScientificReportRevisionV1' / 'negative_control_refusals'
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Checkpoint files written by the generation notebooks (11, 13, 14)
MEDRAG_CORPUS   = 'T21_2026_09_02_ARTICLES'
PQA_CORPUS_DATE = '2026-09-02_articles'
ANS_DIRECT = OUT_DIR / 'direct_answers_negative.json'
ANS_MEDRAG = OUT_DIR / f'medrag_answers_negative_{MEDRAG_CORPUS.lower()}.json'
ANS_PQA    = OUT_DIR / f'paperqa_answers_negative_{PQA_CORPUS_DATE}.json'

# (key, column label) - same model order as the six-method benchmark (notebook 7)
MODELS = [
    ('t21',      'T21 RAG'),
    ('nemotron', 'Nemotron'),
    ('gpt',      'GPT-5.4'),
    ('claude',   'Claude'),
    ('medrag',   'MedRAG'),
    ('paperqa',  'PaperQA2'),
]
MODEL_KEYS = [k for k, _ in MODELS]

_ERR_RE = re.compile(r'\[[A-Z][A-Z0-9_]*_ERROR\]|too many concurrent', re.I)

print('Output dir :', OUT_DIR)
for _p in (ANS_DIRECT, ANS_MEDRAG, ANS_PQA):
    print(f'  {_p.name:55s}', 'found' if _p.exists() else 'MISSING')


Output dir : /path/to/project/T21_research_assistant_v7/analysis/output/ScientificReportRevisionV1/negative_control_refusals
  direct_answers_negative.json                            found
  medrag_answers_negative_t21_2026_09_02_articles.json    found
  paperqa_answers_negative_2026-09-02_articles.json       found


## Cell 2 - The 30 negative-control questions

In [5]:
# ---- Cell 2: the 30 negative-control questions ----
df_q = pd.read_excel(EXCEL_PATH, sheet_name='Questions_for_testing')
df_q.columns = [c.strip() for c in df_q.columns]
neg = df_q[df_q['CTRL_Type'].str.strip().str.lower() == 'negative'].copy().reset_index(drop=True)
neg = neg.rename(columns={'Original question': 'question'})
neg['question'] = neg['question'].astype(str).str.strip()
neg['sl_no'] = neg['SLNo'].astype(str).str.strip().str.replace(r'^[Qq]', '', regex=True)
assert len(neg) == 30, f'expected 30 negative controls, found {len(neg)}'
print(f'negative controls: {len(neg)}')
neg[['sl_no', 'question']]


negative controls: 30


,sl_no,question
0,31,How do the PAM50 subtypes in Down syndrome aff...
1,32,What are some drugs that I can use to treat Hy...
2,33,How do pine trees relate to the development of...
3,34,Do people with Down syndrome have super human ...
4,35,How does the gene FX84A9 drive co-occurring co...
5,36,How does chloroplast dysfunction affect co-occ...
6,37,Why do all people with Down syndrome have the ...
7,38,Can Ivermectin be used to treat Down syndrome?
8,39,Are people with Down syndrome more prone to An...
9,40,What behavioral assays are used to study Down ...


## Cell 3 - Load all six arms from the checkpoints

Reads the checkpoint files written by notebooks 11, 13 and 14 and reports
coverage per arm. A missing file produces a warning and an empty column; an
incomplete file lists the missing SLNos so you know which generation notebook
to re-run.

In [6]:
# ---- Cell 3: load responses ----
_DIRECT_KEYS = ('t21', 'gpt', 'claude', 'nemotron')

if ANS_DIRECT.exists():
    direct_answers = json.loads(ANS_DIRECT.read_text())
    for k in _DIRECT_KEYS:
        neg[f'{k}_answer'] = [direct_answers.get(s, {}).get(f'{k}_answer', '') for s in neg['sl_no']]
else:
    print(f'[WARN] {ANS_DIRECT.name} not found - run notebook 11 first.')
    for k in _DIRECT_KEYS:
        neg[f'{k}_answer'] = ''

if ANS_MEDRAG.exists():
    medrag_answers = json.loads(ANS_MEDRAG.read_text())
    neg['medrag_answer'] = [medrag_answers.get(s, {}).get('answer', '') for s in neg['sl_no']]
else:
    print(f'[WARN] {ANS_MEDRAG.name} not found - run notebook 13 first.')
    neg['medrag_answer'] = ''

if ANS_PQA.exists():
    pqa_answers = json.loads(ANS_PQA.read_text())
    neg['paperqa_answer'] = [pqa_answers.get(s, {}).get('answer', '') for s in neg['sl_no']]
else:
    print(f'[WARN] {ANS_PQA.name} not found - run notebook 14 first.')
    neg['paperqa_answer'] = ''

_SOURCE_NB = {'t21': '11', 'gpt': '11', 'claude': '11', 'nemotron': '11',
              'medrag': '13', 'paperqa': '14'}
print('Coverage per arm:')
for k, lbl in MODELS:
    vals = list(neg[f'{k}_answer'])
    n_ok  = sum(1 for v in vals if v and not _ERR_RE.search(v))
    _bad  = [s for s, v in zip(neg['sl_no'], vals) if not v or _ERR_RE.search(v)]
    line = f'  {lbl:9s} {n_ok}/30'
    if _bad:
        line += f'  incomplete SLNo: {", ".join(_bad)} (re-run notebook {_SOURCE_NB[k]})'
    print(line)


Coverage per arm:
  T21 RAG   30/30
  Nemotron  30/30
  GPT-5.4   30/30
  Claude    30/30
  MedRAG    30/30
  PaperQA2  30/30


## Cell 4 - Merge and save

One Excel sheet (All_responses in negative_control_responses_merged.xlsx):
30 question rows, six model columns (T21 RAG, Nemotron, GPT-5.4, Claude,
MedRAG, PaperQA2), each cell the full response text. A CSV copy and the
timestamped responses_negative_latest.csv/json checkpoint set are saved
alongside.

In [7]:
# ---- Cell 4: merge and save ----
_label_map = dict(MODELS)

# The merged grid: rows = questions, columns = models, cells = responses
df_all = neg[['sl_no', 'question']].copy()
for k, lbl in MODELS:
    df_all[lbl] = neg[f'{k}_answer']

RESULTS_XLSX = OUT_DIR / 'negative_control_responses_merged.xlsx'
with pd.ExcelWriter(RESULTS_XLSX, engine='openpyxl') as xw:
    df_all.to_excel(xw, sheet_name='All_responses', index=False)
    # Widen the question and response columns for readability (openpyxl engine)
    from openpyxl.utils import get_column_letter
    ws = xw.sheets['All_responses']
    ws.column_dimensions['B'].width = 45          # question
    for _ci in range(3, 3 + len(MODELS)):         # model columns C..H
        ws.column_dimensions[get_column_letter(_ci)].width = 60
df_all.to_csv(RESULTS_XLSX.with_suffix('.csv'), index=False, encoding='utf-8-sig')

# Timestamped merged checkpoint (same format the positive-control side uses)
_ts = datetime.now().strftime('%Y%m%d_%H%M%S')
_cols = ['sl_no', 'question'] + [f'{k}_answer' for k in MODEL_KEYS]
df_out = neg[_cols].copy()
_ts_csv = OUT_DIR / f'responses_negative_{_ts}.csv'
df_out.to_csv(_ts_csv, index=False, encoding='utf-8-sig')
shutil.copy2(_ts_csv, OUT_DIR / 'responses_negative_latest.csv')

print(f'saved -> {RESULTS_XLSX.name} (single All_responses sheet, '
      f'{df_all.shape[0]} questions x {len(MODELS)} model columns)')
print(f'saved -> {RESULTS_XLSX.with_suffix(".csv").name}')
print('saved -> responses_negative_latest.csv')
df_all.head(3)


saved -> negative_control_responses_merged.xlsx (single All_responses sheet, 30 questions x 6 model columns)
saved -> negative_control_responses_merged.csv
saved -> responses_negative_latest.csv


,sl_no,question,T21 RAG,Nemotron,GPT-5.4,Claude,MedRAG,PaperQA2
0,31,How do the PAM50 subtypes in Down syndrome aff...,The INCLUDE Publication Corpus does not contai...,The PAM50 assay classifies breast tumors into ...,PAM50 is a breast cancer gene-expression class...,The PAM50 intrinsic subtype classification sys...,"Based on the provided documents, there is no i...",I cannot answer this question due to insuffici...
1,32,What are some drugs that I can use to treat Hy...,Hypothyroidism in Down syndrome is primarily t...,"Levothyroxine sodium, the synthetic T4 hormone...",Hypothyroidism in people with Down syndrome is...,Hypothyroidism is one of the most common comor...,"Based on the provided documents, the primary d...","Levothyroxine (L‑thyroxine, synthetic T4) is t..."
2,33,How do pine trees relate to the development of...,The INCLUDE Publication Corpus does not contai...,There is currently no scientific evidence link...,Pine trees do not have a known causal or devel...,Pine trees do not have any established or reco...,The provided documents do not mention any rela...,I cannot answer this question due to insuffici...


---

Outputs written to analysis/output/ScientificReportRevisionV1/negative_control_refusals/:

- negative_control_responses_merged.xlsx - ONE All_responses sheet: 30
  question rows x 6 model columns, each cell one full response
- negative_control_responses_merged.csv - same table as CSV
- responses_negative_latest.csv (plus a timestamped copy) - merged checkpoint

The refusal classification, counts workbook
(negative_control_refusal_results.xlsx) and figure were produced separately
and are unchanged by this notebook.